# Setup
Das Notebook dient dazu die Images zu downloaden.  
Wichtig hierfür sind in der Config unteranderem MAX_WORKER und die Downloadgeschwindigkeit liegt an:
- MAX_WORKERS COUNT
- Rechenleistung
- Internetverbindung
- Cloud Mangagment (emfohlen lokale Speicherung)

In [ ]:
import os
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
import requests
from PIL import Image
from tqdm import tqdm


PROJECT_ROOT = next(d for d in (Path.cwd(), *Path.cwd().parents)
                    if (d / "config.yaml").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config
from src.mapillary import get_session, load_token

CFG = load_config(PROJECT_ROOT)
CONFIG_PATH = PROJECT_ROOT / "config.yaml"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DATA_PATH_META = PROCESSED_DIR / "metadata.parquet"
IMAGE_PATH = Path(CFG["img_download_path"]).expanduser()
IMG_SIZE = CFG["download_image_size"]
metadata = pd.read_parquet(DATA_PATH_META)
download_metadata = metadata[metadata["split"].isin(["train","database", "query"])].copy()

# Set Max Workers -> Recommendation for new Laptop 96
MAX_WORKERS = CFG["download_workers"]
if MAX_WORKERS == "auto":
    MAX_WORKERS = 96
else:
    MAX_WORKERS = int(MAX_WORKERS)


TOKEN = load_token(PROJECT_ROOT)


print("Download Location: ", IMAGE_PATH)
print(f"Download Threads: {MAX_WORKERS}")
print(f"Zu ladende Bilder: {len(download_metadata):,}")

# API Anfrage
  
Erstelle session, frage API an 

In [ ]:
def download_image(image_id):

    image_id = str(image_id)
    image_file = IMAGE_PATH / f"{image_id}.jpg"

    # Bereits vorhandenes, nicht-leeres Bild überspringen
    if image_file.exists() and image_file.stat().st_size > 0:
        return "exists"

    session = get_session(MAX_WORKERS)

    try:
        api_url = f"https://graph.mapillary.com/{image_id}"

        params = {
            "fields": f"id,thumb_{IMG_SIZE}_url",
            "access_token": TOKEN,
        }

        api_response = session.get(
            api_url,
            params=params,
            timeout=(10, 30),
        )

        api_response.raise_for_status()

        data = api_response.json()

        image_url = data.get(f"thumb_{IMG_SIZE}_url")

        if not image_url:
            print(f"Keine Bild-URL für {image_id}")
            return "failed"

        image_response = session.get(image_url, timeout=(10, 60))
        image_response.raise_for_status()

        if not image_response.content:
            print(f"Leere Bildantwort für {image_id}")
            return "failed"

        image_file.write_bytes(image_response.content)

        return "downloaded"

    except requests.RequestException as e:
        status = e.response.status_code if e.response is not None else "keine Antwort"

        print(f"Fehler bei {image_id}: {type(e).__name__}: {status}")

        return "failed"

    except (ValueError, KeyError) as e:
        print(f"Ungültige Mapillary-Antwort für {image_id}: {type(e).__name__}")

        return "failed"

    except Exception as e:
        # Fängt unerwartete Fehler ab, damit ein einzelnes Bild
        # nicht den gesamten 98k-Download stoppt.
        print(f"Unerwarteter Fehler bei {image_id}: {type(e).__name__}: {e}")

        return "failed"


# Download

In [ ]:

# Ein Verzeichnisdurchlauf statt 332.868 einzelner Abfragen im Threadpool:
# beim wiederholten Lauf ist praktisch alles schon da.
vorhandene_ids = {
    eintrag.name[:-4]
    for eintrag in os.scandir(IMAGE_PATH)
    if eintrag.name.endswith(".jpg") and eintrag.stat().st_size > 0
}

alle_ids = [str(i) for i in download_metadata["image_id"]]
fehlende_ids = [i for i in alle_ids if i not in vorhandene_ids]

print(f"Bereits vorhanden: {len(alle_ids) - len(fehlende_ids):,}")
print(f"Zu holen:          {len(fehlende_ids):,}")

results = {}

if fehlende_ids:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {
            executor.submit(download_image, image_id): image_id
            for image_id in fehlende_ids
        }

        for future in tqdm(as_completed(futures), total=len(futures), desc="Bilder herunterladen"):
            image_id = futures[future]
            try:
                results[image_id] = future.result()
            except Exception as e:
                print(f"Unerwarteter Fehler bei {image_id}: {type(e).__name__}: {e}")
                results[image_id] = "failed"

# Nur diese muessen anschliessend auf Unversehrtheit geprueft werden.
neu_geladene_ids = [i for i, r in results.items() if r == "downloaded"]
failed_ids = [i for i, r in results.items() if r == "failed"]
image_files = list(IMAGE_PATH.glob("*.jpg"))
FAILED_IMAGE_PATH = PROCESSED_DIR / "failed_image_download.txt"
FAILED_IMAGE_PATH.write_text("\n".join(map(str, failed_ids)))


# Auswertung erster Download
print("=" * 50)
print("DOWNLOAD ERGEBNISSE")
print("=" * 50)
print(f"Neu geladen:                {len(neu_geladene_ids):,}")
print(f"Bereits vorhanden:          {len(alle_ids) - len(fehlende_ids):,}")
print(f"Fehlgeschlagene:            {list(results.values()).count('failed'):,}")
print("=" * 50)
print(f"Fehlgeschlagene Bilder:     {len(failed_ids):,}")
print(f"Liste gespeichert unter:    {FAILED_IMAGE_PATH}")
print(f"Lokale JPGs:                {len(image_files):,}")
print("=" * 50)


# Kaputte Bilder

In [ ]:
# Abgebrochene Downloads erkennen. Nur die Bilder aus diesem Lauf -- an den
# uebrigen kann sich nichts geaendert haben. PRUEFE_ALLE liest den gesamten
# Bestand, was bei 330k Dateien etliche Minuten dauert.

PRUEFE_ALLE = False

if PRUEFE_ALLE:
    zu_pruefen = list(IMAGE_PATH.glob("*.jpg"))
else:
    zu_pruefen = [IMAGE_PATH / f"{i}.jpg" for i in neu_geladene_ids]

bad = []

for img in tqdm(zu_pruefen, desc="Bilder pruefen"):
    try:
        with Image.open(img) as im:
            im.verify()
    except Exception:
        bad.append(img)

print(f"Geprueft: {len(zu_pruefen):,}   kaputt: {len(bad)}")

if bad:
    for img in bad:
        img.unlink()
    print(
        f"{len(bad)} geloescht -- die Download-Zelle erneut ausfuehren, "
        "dann werden sie neu geholt."
    )
